# 🎵 Análisis de Audio con Transformada de Fourier

Este notebook realiza los siguientes pasos:

1. **Convierte** un archivo de audio (cualquier formato compatible) a **MP3**
2. **Carga** el archivo MP3 resultante
3. **Aplica la Transformada de Fourier (FFT)** para descomponer la señal en sus frecuencias
4. **Visualiza** el espectro de frecuencias en una gráfica

---

### ¿Qué es la Transformada de Fourier?

La **Transformada de Fourier** es una operación matemática que descompone una señal en el tiempo (como el audio) en sus **componentes de frecuencia**. Básicamente responde la pregunta: *¿cuánto de cada frecuencia contiene esta señal?*

- El **dominio del tiempo** muestra cómo varía la amplitud de la señal a lo largo del tiempo.
- El **dominio de la frecuencia** (resultado de la FFT) muestra qué frecuencias están presentes y con qué intensidad.

Para audio digital usamos la **FFT** (*Fast Fourier Transform*), una versión computacionalmente eficiente de la DFT.

## 📦 Celda 1 — Instalación de dependencias

Instalamos las librerías necesarias:

| Librería | Uso |
|----------|-----|
| `pydub` | Manipulación y conversión de audio |
| `numpy` | Cálculo numérico (FFT) |
| `matplotlib` | Visualización de la gráfica |
| `scipy` | Lectura de archivos WAV y función FFT alternativa |
| `ffmpeg` | Herramienta del sistema requerida por pydub para convertir formatos |

> **Nota:** `ffmpeg` debe estar instalado en el sistema. En Google Colab se instala con `apt-get`.

In [ ]:
# Instalar dependencias de Python
!pip install pydub numpy matplotlib scipy --quiet

# Instalar ffmpeg (necesario para que pydub pueda convertir formatos de audio)
# En sistemas Linux/Colab:
!apt-get install -y ffmpeg --quiet

print("✅ Dependencias instaladas correctamente.")

## 📥 Celda 2 — Importación de librerías

Cargamos todos los módulos que usaremos:

- `AudioSegment` de `pydub`: carga y convierte archivos de audio.
- `numpy`: operaciones matemáticas y la FFT (`np.fft.fft`).
- `matplotlib.pyplot`: creación de gráficas.
- `scipy.io.wavfile`: lectura de archivos WAV para obtener los datos crudos de la señal.
- `os`: manejo de rutas de archivos.

In [ ]:
from pydub import AudioSegment
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.io import wavfile
import os
import warnings

warnings.filterwarnings('ignore')  # Suprimir advertencias menores de scipy

print("✅ Librerías importadas correctamente.")

## 🎧 Celda 3 — Configuración: ruta del archivo de audio

Aquí defines la ruta del archivo de audio original que deseas convertir y analizar.

**Formatos soportados** (gracias a `ffmpeg`): `.wav`, `.mp4`, `.ogg`, `.flac`, `.aac`, `.m4a`, `.mp3`, entre otros.

> ⚠️ Modifica la variable `INPUT_AUDIO_PATH` con la ruta real de tu archivo.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN — Modifica estas rutas según tu archivo
# ─────────────────────────────────────────────────────────────

INPUT_AUDIO_PATH = "mi_audio.wav"      # Ruta del archivo de entrada (cualquier formato)
OUTPUT_MP3_PATH  = "audio_salida.mp3"  # Nombre del archivo MP3 que se generará
OUTPUT_WAV_TEMP  = "audio_temp.wav"    # Archivo WAV temporal para leer los datos crudos

# Verificar que el archivo de entrada existe
if not os.path.exists(INPUT_AUDIO_PATH):
    print(f"⚠️  Archivo '{INPUT_AUDIO_PATH}' no encontrado.")
    print("   Asegúrate de subir el archivo al entorno antes de continuar.")
else:
    print(f"✅ Archivo encontrado: '{INPUT_AUDIO_PATH}'")

## 🔄 Celda 4 — Conversión del audio a MP3

Usamos `pydub.AudioSegment` para:

1. **Cargar** el archivo de audio original con `AudioSegment.from_file()`. Este método detecta automáticamente el formato según la extensión del archivo.
2. **Exportar** el audio en formato MP3 con `export()`, especificando el `format="mp3"`.

Internamente, `pydub` delega la codificación/decodificación a **ffmpeg**, que es el motor real de conversión.

También exportamos una copia en WAV sin comprimir, que usaremos más adelante para leer los datos numéricos de la señal (ya que WAV es más fácil de parsear directamente que MP3).

In [ ]:
# Cargar el archivo de audio original (pydub detecta el formato automáticamente)
print(f"🔄 Cargando '{INPUT_AUDIO_PATH}'...")
audio = AudioSegment.from_file(INPUT_AUDIO_PATH)

# Mostrar información del audio cargado
print(f"   ├─ Canales       : {audio.channels} ({'Mono' if audio.channels == 1 else 'Estéreo'})")
print(f"   ├─ Tasa de muestreo: {audio.frame_rate} Hz")
print(f"   ├─ Duración       : {len(audio) / 1000:.2f} segundos")
print(f"   └─ Bits por muestra: {audio.sample_width * 8} bits")

# Exportar a MP3
print(f"\n💾 Exportando a MP3 → '{OUTPUT_MP3_PATH}'...")
audio.export(OUTPUT_MP3_PATH, format="mp3")
print(f"✅ Archivo MP3 generado: '{OUTPUT_MP3_PATH}'")

# Exportar también a WAV para poder leer los datos crudos con scipy
audio.export(OUTPUT_WAV_TEMP, format="wav")
print(f"✅ Archivo WAV temporal generado: '{OUTPUT_WAV_TEMP}'")

## 📊 Celda 5 — Lectura de la señal y conversión a array numérico

Para aplicar la FFT necesitamos los **valores numéricos** de la señal de audio (amplitudes en función del tiempo).

Usamos `scipy.io.wavfile.read()` que nos devuelve:
- `sample_rate`: la **tasa de muestreo** en Hz (número de muestras por segundo, ej. 44100 Hz).
- `data`: un array NumPy con las **amplitudes** de la señal.

Si el audio es **estéreo** (2 canales), promediamos ambos canales para obtener una señal mono, lo que simplifica el análisis.

El array resultante se **normaliza** al rango `[-1.0, 1.0]` dividiendo por el valor máximo posible según la profundidad de bits.

In [ ]:
# Leer el archivo WAV con scipy para obtener los datos crudos
sample_rate, data = wavfile.read(OUTPUT_WAV_TEMP)

print(f"📌 Tasa de muestreo : {sample_rate} Hz")
print(f"📌 Forma del array  : {data.shape}")
print(f"📌 Tipo de dato     : {data.dtype}")

# Si es estéreo (2 columnas), convertir a mono promediando los canales
if data.ndim == 2:
    print("🔀 Audio estéreo detectado → promediando canales para obtener mono")
    data = data.mean(axis=1)

# Normalizar la señal al rango [-1.0, 1.0]
# El factor de normalización depende del tipo de dato (8, 16 o 32 bits)
max_val = np.iinfo(data.dtype).max if np.issubdtype(data.dtype, np.integer) else 1.0
signal = data.astype(np.float64) / max_val

# Crear el eje de tiempo en segundos
duration = len(signal) / sample_rate
time_axis = np.linspace(0, duration, len(signal))

print(f"\n✅ Señal lista:")
print(f"   ├─ Total de muestras : {len(signal):,}")
print(f"   ├─ Duración          : {duration:.3f} s")
print(f"   └─ Rango normalizado : [{signal.min():.3f}, {signal.max():.3f}]")

## 🔬 Celda 6 — Aplicación de la Transformada de Fourier (FFT)

### ¿Cómo funciona la FFT?

La **Fast Fourier Transform** toma `N` muestras en el dominio del tiempo y devuelve `N` números complejos en el dominio de la frecuencia.

**Pasos del cálculo:**

1. `np.fft.fft(signal)` → calcula la FFT y devuelve coeficientes complejos.
2. `np.abs(fft_result)` → tomamos el **módulo** (magnitud) de cada coeficiente complejo. Esto nos dice qué tan fuerte está presente cada frecuencia.
3. Solo nos quedamos con la **primera mitad** del resultado (`N//2`), ya que la FFT de una señal real es simétrica.
4. `np.fft.fftfreq(N, d=1/sample_rate)` → genera el **eje de frecuencias** en Hz.

### Ventaneo (Windowing)

Antes de aplicar la FFT, multiplicamos la señal por una **ventana de Hann**. Esto reduce el efecto de *spectral leakage* (fuga espectral), que ocurre cuando la señal no es perfectamente periódica en el intervalo analizado.

In [ ]:
N = len(signal)  # Número total de muestras

# ── Aplicar ventana de Hann para reducir spectral leakage ──────────────────
window   = np.hanning(N)           # Ventana de Hann: suaviza los bordes de la señal
signal_w = signal * window         # Señal ventaneada

# ── Calcular la FFT ────────────────────────────────────────────────────────
fft_result = np.fft.fft(signal_w)  # Array de N números complejos

# ── Magnitud del espectro ──────────────────────────────────────────────────
# |X[k]| da la amplitud de cada componente de frecuencia
# Multiplicamos por 2/N para normalizar (y x2 por quedarnos con mitad)
magnitude = (2 / N) * np.abs(fft_result[:N // 2])

# ── Eje de frecuencias ─────────────────────────────────────────────────────
# fftfreq genera las frecuencias correspondientes a cada bin de la FFT
# d = 1/sample_rate es el intervalo de muestreo en segundos
frequencies = np.fft.fftfreq(N, d=1 / sample_rate)[:N // 2]

# ── Frecuencia dominante ───────────────────────────────────────────────────
idx_max   = np.argmax(magnitude)
freq_dom  = frequencies[idx_max]
mag_max   = magnitude[idx_max]

print(f"✅ FFT calculada sobre {N:,} muestras")
print(f"   ├─ Resolución frecuencial : {sample_rate/N:.4f} Hz por bin")
print(f"   ├─ Frecuencia máxima repr.: {frequencies[-1]:.1f} Hz (Nyquist)")
print(f"   └─ Frecuencia dominante   : {freq_dom:.2f} Hz (magnitud {mag_max:.4f})")

## 📈 Celda 7 — Visualización del espectro de frecuencias

Generamos dos gráficas:

1. **Señal en el tiempo**: Muestra la forma de onda (amplitud vs tiempo). Permite ver si el audio tiene silenciosos, picos, etc.
2. **Espectro de frecuencias (FFT)**: Muestra qué frecuencias están presentes y con qué intensidad. Limitamos el eje X a **20,000 Hz**, que es el límite de la audición humana.

En el espectro marcamos la **frecuencia dominante** (el pico más alto) con una línea punteada roja.

In [ ]:
# Limitar la visualización al rango audible humano (20 Hz – 20,000 Hz)
freq_max_plot = min(20000, sample_rate / 2)
mask = frequencies <= freq_max_plot

# ── Configuración estética ─────────────────────────────────────────────────
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 1, figsize=(14, 9), facecolor='#0d0d0d')
fig.suptitle('Análisis de Audio — Transformada de Fourier (FFT)',
             fontsize=16, fontweight='bold', color='white', y=0.98)

# ── Gráfica 1: Señal en el tiempo ─────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#111111')

# Submuestreamos para no graficar millones de puntos
step = max(1, len(signal) // 10000)
ax1.plot(time_axis[::step], signal[::step],
         color='#00e5ff', linewidth=0.6, alpha=0.85)
ax1.fill_between(time_axis[::step], signal[::step],
                 alpha=0.15, color='#00e5ff')

ax1.set_title('Forma de onda — Dominio del tiempo', fontsize=13,
              color='#cccccc', pad=10)
ax1.set_xlabel('Tiempo (segundos)', color='#aaaaaa', fontsize=11)
ax1.set_ylabel('Amplitud normalizada', color='#aaaaaa', fontsize=11)
ax1.set_xlim([0, duration])
ax1.set_ylim([-1.05, 1.05])
ax1.tick_params(colors='#888888')
ax1.spines[:].set_color('#333333')
ax1.grid(True, color='#2a2a2a', linewidth=0.5)

# ── Gráfica 2: Espectro de frecuencias (FFT) ──────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#111111')

ax2.plot(frequencies[mask], magnitude[mask],
         color='#ff6b35', linewidth=0.8, alpha=0.9)
ax2.fill_between(frequencies[mask], magnitude[mask],
                 alpha=0.25, color='#ff6b35')

# Marcar la frecuencia dominante
ax2.axvline(x=freq_dom, color='#ffdd00', linewidth=1.2,
            linestyle='--', alpha=0.8,
            label=f'Frecuencia dominante: {freq_dom:.1f} Hz')
ax2.annotate(f'{freq_dom:.1f} Hz',
             xy=(freq_dom, mag_max),
             xytext=(freq_dom + freq_max_plot * 0.04, mag_max * 0.85),
             color='#ffdd00', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='#ffdd00', lw=1.2))

ax2.set_title('Espectro de frecuencias — Dominio de la frecuencia (FFT)',
              fontsize=13, color='#cccccc', pad=10)
ax2.set_xlabel('Frecuencia (Hz)', color='#aaaaaa', fontsize=11)
ax2.set_ylabel('Magnitud', color='#aaaaaa', fontsize=11)
ax2.set_xlim([0, freq_max_plot])
ax2.tick_params(colors='#888888')
ax2.spines[:].set_color('#333333')
ax2.grid(True, color='#2a2a2a', linewidth=0.5)
ax2.legend(facecolor='#1a1a1a', edgecolor='#444444',
           labelcolor='#ffdd00', fontsize=10)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{int(x):,} Hz'))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('espectro_fourier.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
plt.show()
print("✅ Gráfica guardada como 'espectro_fourier.png'")

## 📋 Celda 8 — Resumen estadístico del espectro

Calculamos algunas métricas útiles sobre el contenido frecuencial del audio:

- **Centroide espectral**: el "centro de masa" del espectro, indica dónde está concentrada la energía.
- **Top 5 frecuencias**: las frecuencias con mayor magnitud (los picos dominantes).
- **Energía en bandas**: cuánta energía hay en las bandas de bajos, medios y agudos.

In [ ]:
# ── Centroide espectral ────────────────────────────────────────────────────
# Es la media ponderada de las frecuencias, donde el peso es la magnitud
centroide = np.sum(frequencies * magnitude) / np.sum(magnitude)

# ── Top 5 frecuencias dominantes ──────────────────────────────────────────
top_indices = np.argsort(magnitude)[::-1][:5]
top_freqs   = frequencies[top_indices]
top_mags    = magnitude[top_indices]

# ── Energía por banda ─────────────────────────────────────────────────────
def energia_banda(f_min, f_max):
    """Calcula la energía total en un rango de frecuencias."""
    mask_b = (frequencies >= f_min) & (frequencies < f_max)
    return np.sum(magnitude[mask_b] ** 2)

e_bajos   = energia_banda(20, 300)
e_medios  = energia_banda(300, 4000)
e_agudos  = energia_banda(4000, 20000)
e_total   = e_bajos + e_medios + e_agudos

# ── Mostrar resultados ─────────────────────────────────────────────────────
print("═" * 52)
print("          RESUMEN DEL ANÁLISIS ESPECTRAL")
print("═" * 52)
print(f"  Tasa de muestreo  : {sample_rate:,} Hz")
print(f"  Duración          : {duration:.3f} s")
print(f"  Frecuencia dominante: {freq_dom:.2f} Hz")
print(f"  Centroide espectral : {centroide:.2f} Hz")
print()
print("  TOP 5 FRECUENCIAS:")
for i, (f, m) in enumerate(zip(top_freqs, top_mags), 1):
    print(f"    {i}. {f:>9.2f} Hz  │  magnitud: {m:.5f}")
print()
print("  ENERGÍA POR BANDA:")
print(f"    Bajos  (  20 – 300 Hz)  : {e_bajos/e_total*100:5.1f}%")
print(f"    Medios ( 300 – 4000 Hz) : {e_medios/e_total*100:5.1f}%")
print(f"    Agudos (4000 – 20000 Hz): {e_agudos/e_total*100:5.1f}%")
print("═" * 52)

## 🧹 Celda 9 — Limpieza de archivos temporales

Eliminamos el archivo WAV temporal que creamos como paso intermedio. El MP3 y la imagen del espectro se conservan.

In [ ]:
# Eliminar archivo WAV temporal
if os.path.exists(OUTPUT_WAV_TEMP):
    os.remove(OUTPUT_WAV_TEMP)
    print(f"🗑️  Archivo temporal '{OUTPUT_WAV_TEMP}' eliminado.")

print("\n📁 Archivos generados:")
for fname in [OUTPUT_MP3_PATH, 'espectro_fourier.png']:
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f"   ✅ {fname}  ({size_kb:.1f} KB)")

---

## 📚 Referencias y conceptos clave

| Concepto | Descripción |
|----------|-------------|
| **FFT** | *Fast Fourier Transform* — algoritmo O(N log N) para calcular la DFT |
| **Tasa de muestreo** | Muestras por segundo (Hz). CD: 44100 Hz, DVD: 48000 Hz |
| **Teorema de Nyquist** | La frecuencia máxima representable es `sample_rate / 2` |
| **Spectral leakage** | Distorsión en el espectro cuando la señal no es periódica en el intervalo |
| **Ventana de Hann** | Función multiplicativa que suaviza los bordes y reduce el leakage |
| **Centroide espectral** | Media ponderada de frecuencias — correlaciona con el "brillo" del sonido |
| **Magnitud FFT** | `|X[k]|` = amplitud de la frecuencia k-ésima |

---
*Notebook generado para análisis de audio con Python · NumPy · SciPy · Pydub · Matplotlib*